# Data Cleaning

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

support_file = Path("SupportData.xlsx")
gl_account_file = Path("GLAccountData.xlsx")
gl_entry_file = Path("GLEntryData.xlsx")

clean_data_dir = Path("outputs") / "cleaned_data"
support_cleaned_file = clean_data_dir / "support_cleaned.csv"
gl_accounts_cleaned_file = clean_data_dir / "gl_accounts_cleaned.csv"
gl_entries_cleaned_file = clean_data_dir / "gl_entries_cleaned.csv"

random_state = 42

def clean_columns(df):
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip().str.replace(".", "_", regex=False).str.replace(" ", "_", regex=False)
    return df

def normalise_key(series):
    return series.astype("string").str.strip().str.upper().str.replace(r"\.0$", "", regex=True)

def normalise_gl_description(series):
    return (series.astype("string").str.upper()
        .str.replace(r"\bCLAIMBACKS?\b", "", regex=True)
        .str.replace(r"\b[A-Z]\d+[A-Z]*\b", "", regex=True)
        .str.replace(r"\bLIMITED\b|\bLTD\b", "", regex=True)
        .str.replace(r"\s+", " ", regex=True).str.strip())

def support_base():
    df = clean_columns(pd.read_excel(support_file, dtype=str))
    df["claim_line_id"] = np.arange(1, len(df) + 1)
    df["claim_date"] = pd.to_datetime(df.get("Date"), errors="coerce", dayfirst=True)
    df["customer_date"] = df["claim_date"]
    df["customer_year"] = df["claim_date"].dt.year
    df["customer_month"] = df["claim_date"].dt.month
    qty = pd.to_numeric(df.get("SalesDeliveryNoteLine_Quantity"), errors="coerce").fillna(pd.to_numeric(df.get("SalesInvoiceLine_Quantity"), errors="coerce")).fillna(0)
    unit = pd.to_numeric(df.get("UnitClaimAmount"), errors="coerce").fillna(0)
    total = pd.to_numeric(df.get("TotalClaimAmount"), errors="coerce")
    df["credit_due_value"] = total.fillna(unit * qty)
    credit_note = df.get("SalesCreditNote_Number", pd.Series("", index=df.index)).astype("string").str.strip().ne("").fillna(False)
    df["credit_received_value"] = np.where(credit_note.to_numpy(dtype=bool), df["credit_due_value"].clip(lower=0), 0)
    df["outstanding_credit_value"] = df["credit_due_value"] - df["credit_received_value"]
    df["recovered_credit_rate"] = df["credit_received_value"].div(df["credit_due_value"].replace(0, np.nan)).fillna(0)
    df["supplier_reference_value"] = df["credit_due_value"]
    df["supplier_reference_match_status"] = "Derived from SupportData.xlsx"
    has_received_credit = df["credit_received_value"].gt(0).fillna(False)
    has_outstanding_credit = df["outstanding_credit_value"].gt(0).fillna(False)
    df["credit_recovery_status"] = np.select(
        [has_received_credit.to_numpy(dtype=bool), has_outstanding_credit.to_numpy(dtype=bool)],
        ["Credit note present", "Outstanding"],
        default="No claim value",
    )
    df["days_outstanding"] = (pd.Timestamp.today().normalize() - df["claim_date"]).dt.days
    df["ageing_band"] = pd.cut(df["days_outstanding"], [-1,30,60,90,180,365,np.inf], labels=["0-30","31-60","61-90","91-180","181-365","365+"])
    df["requires_review"] = has_outstanding_credit
    df["review_reason"] = np.where(has_outstanding_credit.to_numpy(dtype=bool), "Outstanding credit value", "")
    return df

def load_gl_context():
    accounts = clean_columns(pd.read_excel(gl_account_file, dtype=str)) if gl_account_file.exists() else pd.DataFrame()
    entries = clean_columns(pd.read_excel(gl_entry_file, dtype=str)) if gl_entry_file.exists() else pd.DataFrame()

    if not accounts.empty:
        accounts["gl_account_code"] = normalise_key(accounts.get("Code", pd.Series(pd.NA,index=accounts.index)))
        accounts["gl_account_description"] = accounts.get("Description", pd.Series(pd.NA,index=accounts.index))
        accounts["gl_supplier_key"] = normalise_gl_description(accounts["gl_account_description"])
        accounts["gl_current_balance"] = pd.to_numeric(accounts.get("CurrentBalance", pd.NA), errors="coerce")
        accounts["gl_current_balance_abs"] = accounts["gl_current_balance"].abs()
    if not entries.empty:
        entries["gl_account_code"] = normalise_key(entries.get("GL_Account_Code", pd.Series(pd.NA,index=entries.index)))
        entries["gl_account_description"] = entries.get("Description", pd.Series(pd.NA,index=entries.index))
        entries["gl_supplier_key"] = normalise_gl_description(entries["gl_account_description"])
        entries["gl_credit_amount"] = pd.to_numeric(entries.get("Credit_Amount", pd.NA), errors="coerce").fillna(0)
        entries["gl_entry_line_date"] = pd.to_datetime(entries.get("Entry_Line_Date", pd.NA), errors="coerce", dayfirst=True)
    if not entries.empty:
        entries["gl_entry_period_month"] = entries["gl_entry_line_date"].dt.to_period("M").astype("string")
        summary = entries.groupby(["gl_account_code","gl_account_description","gl_supplier_key"], dropna=False).agg(
            gl_entry_rows=("gl_account_code","size"),
            gl_total_credit_amount=("gl_credit_amount","sum"),
            gl_first_entry_date=("gl_entry_line_date","min"),
            gl_last_entry_date=("gl_entry_line_date","max"),
        ).reset_index()
        if not accounts.empty:
            account_cols = [col for col in ["gl_account_code", "gl_current_balance", "gl_current_balance_abs"] if col in accounts.columns]
            summary = summary.merge(accounts[account_cols].drop_duplicates("gl_account_code"), on="gl_account_code", how="left")
    else:
        summary = pd.DataFrame()
    return accounts, entries, summary

df = support_base()
gl_accounts_clean, gl_entries_clean, gl_account_summary_clean = load_gl_context()
gl_accounts = gl_accounts_clean
gl_entries = gl_entries_clean
gl_account_summary = gl_account_summary_clean
supplier_gl_payment_model_base = pd.DataFrame()
print("Loaded and cleaned raw support data:", df.shape)
print("GL accounts:", gl_accounts_clean.shape, "GL entries:", gl_entries_clean.shape)
display(df.head())
display(gl_account_summary_clean.head())

Loaded and cleaned raw support data: (73320, 57)
GL accounts: (21, 9) GL entries: (170, 12)


,Date,Month,Year,SalesDeliveryNote_Branch,Customer,Customer_Name,Product,Product_ManufacturerProductCode,Contract_Number,SourceTransactionType,Contract_D_ContractNumber,SalesInvoice_Number,SalesCreditNote_Number,SalesDeliveryNote_Number,SalesReturnNote_Number,SalesOrder_Number,Status,UnitClaimAmount,TotalClaimAmount,SalesDeliveryNoteLine_Quantity,SalesInvoiceLine_Quantity,SalesInvoice_Branch,SalesDeliveryNoteLine_DiscountPercentage,Product_Category,SalesCreditNoteLine_D_InvoiceCost,SalesCreditNoteLine_D_FixedCost,SalesDeliveryNoteLine_D_InvoiceCost,SalesDeliveryNoteLine_D_FixedCost,SalesInvoiceLine_D_InvoiceCost,SalesInvoiceLine_D_FixedCost,SalesDeliveryNote_Branch_1,SalesDeliveryNote_NetAmountLessDiscountBase,Product_CSQL_InvoiceCostForbranch,Product_CSQL_ListPriceForBranch,Product_CALC_FixedCostForBranch,Contract_Description,Contract_Expression,Contract_ValidFrom,Contract_ValidTo,Contract_Supplier_Code,Contract_Supplier_Name,claim_line_id,claim_date,customer_date,customer_year,customer_month,credit_due_value,credit_received_value,outstanding_credit_value,recovered_credit_rate,supplier_reference_value,supplier_reference_match_status,credit_recovery_status,days_outstanding,ageing_band,requires_review,review_reason
0,26/02/2024,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS1228,38300,0086,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,1.366861,8.2012,6,0,NaN,69,PRSCU,0,0,5.4894,2.6953,0,0,02,392.18,6.235348,14.21,4.61415752,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,1,2024-02-26,2024-02-26,2024,2,8.2012,0.0,8.2012,0.0,8.2012,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
1,26/02/2024,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS12S28,38322,0086,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,1.783139,3.5663,2,0,NaN,69,PRSCU,0,0,7.1612,3.5162,0,0,02,392.18,8.130964,18.53,6.01691336,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,2,2024-02-26,2024-02-26,2024,2,3.5663,0.0,3.5663,0.0,3.5663,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
2,26/02/2024,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS62822,38204,0086,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,1.190942,2.3819,2,0,NaN,69,PRSCU,0,0,4.7829,2.3484,0,0,02,392.18,5.432344,12.38,4.01993456,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,3,2024-02-26,2024-02-26,2024,2,2.3819,0.0,2.3819,0.0,2.3819,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
3,26/02/2024,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS252815,38492,0086,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,2.488979,9.9559,4,0,NaN,69,PRSCU,0,0,9.9959,4.908,0,0,02,392.18,11.351756,25.87,8.40029944,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,4,2024-02-26,2024-02-26,2024,2,9.9559,0.0,9.9559,0.0,9.9559,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
4,26/02/2024,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS252215,38490,0086,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,0.890474,5.3428,6,0,NaN,69,PRSCU,0,0,3.5762,1.7559,0,0,02,392.18,4.0589,9.25,3.003586,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,5,2024-02-26,2024-02-26,2024,2,5.3428,0.0,5.3428,0.0,5.3428,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value


,gl_account_code,gl_account_description,gl_supplier_key,gl_entry_rows,gl_total_credit_amount,gl_first_entry_date,gl_last_entry_date,gl_current_balance,gl_current_balance_abs
0,01-85001,Supplier13 E071 Claimbacks,SUPPLIER13,105,242784.06,2026-01-05,2026-03-07,-242784.06,242784.06
1,01-85002,Supplier2 E005 Claimbacks,SUPPLIER2,1,6240.00,NaT,NaT,-6240.00,6240.00
2,01-85007,Supplier9 E022 Claimbacks,SUPPLIER9,5,5774.89,2026-04-06,2026-10-07,-5774.89,5774.89
3,01-85017,Supplier3 E055 Claimbacks,SUPPLIER3,4,5550.00,2026-10-04,2026-10-04,-5550.00,5550.00
4,01-85021,Supplier19 E070 Claimbacks,SUPPLIER19,4,9124.28,NaT,NaT,-9124.28,9124.28


Missingness table formula for the merged dataset.

In [2]:
def missingness_table(df):
    result = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": df.isna().mean() * 100,
        "data_type": df.dtypes.astype(str)
    })
    
    return result.sort_values("missing_percentage", ascending=False)

In [3]:
print("Model base is loaded from SupportData.xlsx in the setup cell.")
display(df.head())


Model base is loaded from SupportData.xlsx in the setup cell.


,Date,Month,Year,SalesDeliveryNote_Branch,Customer,Customer_Name,Product,Product_ManufacturerProductCode,Contract_Number,SourceTransactionType,Contract_D_ContractNumber,SalesInvoice_Number,SalesCreditNote_Number,SalesDeliveryNote_Number,SalesReturnNote_Number,SalesOrder_Number,Status,UnitClaimAmount,TotalClaimAmount,SalesDeliveryNoteLine_Quantity,SalesInvoiceLine_Quantity,SalesInvoice_Branch,SalesDeliveryNoteLine_DiscountPercentage,Product_Category,SalesCreditNoteLine_D_InvoiceCost,SalesCreditNoteLine_D_FixedCost,SalesDeliveryNoteLine_D_InvoiceCost,SalesDeliveryNoteLine_D_FixedCost,SalesInvoiceLine_D_InvoiceCost,SalesInvoiceLine_D_FixedCost,SalesDeliveryNote_Branch_1,SalesDeliveryNote_NetAmountLessDiscountBase,Product_CSQL_InvoiceCostForbranch,Product_CSQL_ListPriceForBranch,Product_CALC_FixedCostForBranch,Contract_Description,Contract_Expression,Contract_ValidFrom,Contract_ValidTo,Contract_Supplier_Code,Contract_Supplier_Name,claim_line_id,claim_date,customer_date,customer_year,customer_month,credit_due_value,credit_received_value,outstanding_credit_value,recovered_credit_rate,supplier_reference_value,supplier_reference_match_status,credit_recovery_status,days_outstanding,ageing_band,requires_review,review_reason
0,26/02/2024,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS1228,38300,0086,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,1.366861,8.2012,6,0,NaN,69,PRSCU,0,0,5.4894,2.6953,0,0,02,392.18,6.235348,14.21,4.61415752,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,1,2024-02-26,2024-02-26,2024,2,8.2012,0.0,8.2012,0.0,8.2012,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
1,26/02/2024,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS12S28,38322,0086,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,1.783139,3.5663,2,0,NaN,69,PRSCU,0,0,7.1612,3.5162,0,0,02,392.18,8.130964,18.53,6.01691336,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,2,2024-02-26,2024-02-26,2024,2,3.5663,0.0,3.5663,0.0,3.5663,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
2,26/02/2024,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS62822,38204,0086,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,1.190942,2.3819,2,0,NaN,69,PRSCU,0,0,4.7829,2.3484,0,0,02,392.18,5.432344,12.38,4.01993456,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,3,2024-02-26,2024-02-26,2024,2,2.3819,0.0,2.3819,0.0,2.3819,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
3,26/02/2024,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS252815,38492,0086,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,2.488979,9.9559,4,0,NaN,69,PRSCU,0,0,9.9959,4.908,0,0,02,392.18,11.351756,25.87,8.40029944,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,4,2024-02-26,2024-02-26,2024,2,9.9559,0.0,9.9559,0.0,9.9559,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value
4,26/02/2024,NaN,NaN,02,M3M500,M3 Mechanical Limited,YXS252215,38490,0086,SL/Del,039-C-0692,NaN,NaN,SO0000413/1,NaN,SO0000413,Claimed,0.890474,5.3428,6,0,NaN,69,PRSCU,0,0,3.5762,1.7559,0,0,02,392.18,4.0589,9.25,3.003586,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,5,2024-02-26,2024-02-26,2024,2,5.3428,0.0,5.3428,0.0,5.3428,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value


In [4]:
print("GL context is loaded from GLAccountData.xlsx and GLEntryData.xlsx in the setup cell.")
display(gl_account_summary_clean.head())


GL context is loaded from GLAccountData.xlsx and GLEntryData.xlsx in the setup cell.


,gl_account_code,gl_account_description,gl_supplier_key,gl_entry_rows,gl_total_credit_amount,gl_first_entry_date,gl_last_entry_date,gl_current_balance,gl_current_balance_abs
0,01-85001,Supplier13 E071 Claimbacks,SUPPLIER13,105,242784.06,2026-01-05,2026-03-07,-242784.06,242784.06
1,01-85002,Supplier2 E005 Claimbacks,SUPPLIER2,1,6240.00,NaT,NaT,-6240.00,6240.00
2,01-85007,Supplier9 E022 Claimbacks,SUPPLIER9,5,5774.89,2026-04-06,2026-10-07,-5774.89,5774.89
3,01-85017,Supplier3 E055 Claimbacks,SUPPLIER3,4,5550.00,2026-10-04,2026-10-04,-5550.00,5550.00
4,01-85021,Supplier19 E070 Claimbacks,SUPPLIER19,4,9124.28,NaT,NaT,-9124.28,9124.28


In [5]:
quality_report = missingness_table(df)

display(quality_report)

,missing_count,missing_percentage,data_type
Month,73320,100.000000,object
Year,73320,100.000000,object
SalesReturnNote_Number,73313,99.990453,object
SalesCreditNote_Number,71579,97.625477,object
SalesInvoice_Number,19169,26.144299,object
SalesInvoice_Branch,19169,26.144299,object
SalesOrder_Number,18465,25.184124,object
SalesDeliveryNote_Number,18464,25.182761,object
SalesDeliveryNote_Branch,18464,25.182761,object
SalesDeliveryNote_Branch_1,18464,25.182761,object


Most missing values contained in various features, most frequently in sales return and credit note numbers. Full missingness in year and month features

In [6]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

if "claim_line_id" in df.columns:
    print("Claim line rows:", len(df))
    print("Distinct claim_line_id values:", df["claim_line_id"].nunique())
    repeated_claim_line_ids = df["claim_line_id"].duplicated(keep=False).sum()
    print("Repeated claim_line_id rows:", repeated_claim_line_ids)
else:
    print("claim_line_id is missing; rerun 0.1 - Data Prep & Generation before cleaning.")

if duplicate_count > 0:
    display(df[df.duplicated(keep=False)].head(20))


Duplicate rows: 0
Claim line rows: 73320
Distinct claim_line_id values: 73320
Repeated claim_line_id rows: 0


In [7]:
df_clean = df.copy()

for col in df_clean.columns:
    if df_clean[col].isna().any():
        df_clean[f"{col}_was_missing"] = df_clean[col].isna().astype(int)

In [8]:

gl_accounts_clean = gl_accounts.copy()
gl_entries_clean = gl_entries.copy()
gl_account_summary_clean = gl_account_summary.copy()

for temp in [gl_accounts_clean, gl_entries_clean, gl_account_summary_clean]:
    if temp.empty:
        continue
    for col in temp.select_dtypes(include=["object", "category", "string"]).columns:
        temp[col] = temp[col].astype("string").fillna("Unknown")
    for col in temp.select_dtypes(include=["number"]).columns:
        temp[col] = temp[col].fillna(0)

if not gl_account_summary_clean.empty:
    gl_account_summary_clean["gl_has_entry_detail"] = gl_account_summary_clean["gl_entry_rows"].fillna(0).gt(0).astype(int)
    display(gl_account_summary_clean.head(10))


,gl_account_code,gl_account_description,gl_supplier_key,gl_entry_rows,gl_total_credit_amount,gl_first_entry_date,gl_last_entry_date,gl_current_balance,gl_current_balance_abs,gl_has_entry_detail
0,01-85001,Supplier13 E071 Claimbacks,SUPPLIER13,105,242784.06,2026-01-05,2026-03-07,-242784.06,242784.06,1
1,01-85002,Supplier2 E005 Claimbacks,SUPPLIER2,1,6240.00,NaT,NaT,-6240.00,6240.00,1
2,01-85007,Supplier9 E022 Claimbacks,SUPPLIER9,5,5774.89,2026-04-06,2026-10-07,-5774.89,5774.89,1
3,01-85017,Supplier3 E055 Claimbacks,SUPPLIER3,4,5550.00,2026-10-04,2026-10-04,-5550.00,5550.00,1
4,01-85021,Supplier19 E070 Claimbacks,SUPPLIER19,4,9124.28,NaT,NaT,-9124.28,9124.28,1
5,01-85036,Supplier8 E175 Claimbacks,SUPPLIER8,2,4782.68,2026-04-06,2026-04-06,-4782.68,4782.68,1
6,01-85046,Supplier10 E234 Claimbacks,SUPPLIER10,8,925.00,2026-08-04,2026-08-04,-925.00,925.00,1
7,01-85056,Supplier17 E269A Claimbacks,SUPPLIER17,1,576.00,NaT,NaT,-576.00,576.00,1
8,01-85079,Supplier18 E555 Claimbacks,SUPPLIER18,11,46476.48,2026-10-06,2026-10-06,-46476.48,46476.48,1
9,01-85115,Supplier20 E948 Claimbacks,SUPPLIER20,5,3431.79,2026-07-04,2026-07-04,-3431.79,3431.79,1


In [9]:
numeric_cols = df_clean.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=["object", "category"]).columns.tolist()
datetime_cols = df_clean.select_dtypes(include=["datetime64[ns]"]).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("Datetime columns:", len(datetime_cols))

Numeric columns: 26
Categorical columns: 45
Datetime columns: 2


In [10]:
zero_fill_cols = [
    "credit_due_value",
    "credit_received_value",
    "supplier_reference_value",
    "outstanding_credit_value",
    "recovered_credit_rate",
    "customer_quantity",
    "invoice_quantity",
    "contract_minimum_quantity",
    "invoice_cost",
    "fixed_cost",
    "list_price",
    "estimated_gross_margin_before_credit",
    "estimated_margin_if_credit_paid",
    "estimated_margin_if_credit_not_paid",
    "margin_at_risk_from_unpaid_credit",
    "margin_after_claim"
]

for col in zero_fill_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").fillna(0)


missing check

In [11]:
for col in categorical_cols:
    df_clean[col] = df_clean[col].astype("object").fillna("Unknown")

if "days_since_claim" in df_clean.columns:
    df_clean["days_since_claim"] = df_clean["days_since_claim"].fillna(-1)

if "ageing_band" in df_clean.columns:
    df_clean["ageing_band"] = df_clean["ageing_band"].astype("object").fillna("Unknown")

missing_after_cleaning = missingness_table(df_clean)

display(missing_after_cleaning)

,missing_count,missing_percentage,data_type
Date,0,0.0,object
Month,0,0.0,object
Year,0,0.0,object
SalesDeliveryNote_Branch,0,0.0,object
Customer,0,0.0,object
Customer_Name,0,0.0,object
Product,0,0.0,object
Product_ManufacturerProductCode,0,0.0,object
Contract_Number,0,0.0,object
SourceTransactionType,0,0.0,object


In [12]:
missing_pct = df_clean.isna().mean()

high_missing_cols = missing_pct[missing_pct > 0.80].index.tolist()

print("Columns with more than 80% missing values:")
print(high_missing_cols)

Columns with more than 80% missing values:
[]


In [13]:
protected_cols = [
    "claim_line_id",
    "claim_line_sequence_within_key",
    "credit_due_value",
    "credit_received_value",
    "outstanding_credit_value",
    "recovered_credit_rate",
    "supplier_reference_value",
    "supplier_reference_match_status",
    "credit_recovery_status",
    "recovery_evidence",
    "requires_review",
    "days_since_claim",
    "days_outstanding",
    "ageing_band",
    "claim_date",
    "margin_at_risk_from_unpaid_credit",
    "margin_risk_review"
]

quarantine_cols = [
    col for col in high_missing_cols
    if col not in protected_cols
]

df_model_base = df_clean.drop(columns=quarantine_cols, errors="ignore")

print("Original shape:", df.shape)
print("Clean model base shape:", df_model_base.shape)
print("Protected columns retained:", [col for col in protected_cols if col in df_model_base.columns])


Original shape: (73320, 57)
Clean model base shape: (73320, 74)
Protected columns retained: ['claim_line_id', 'credit_due_value', 'credit_received_value', 'outstanding_credit_value', 'recovered_credit_rate', 'supplier_reference_value', 'supplier_reference_match_status', 'credit_recovery_status', 'requires_review', 'days_outstanding', 'ageing_band', 'claim_date']


In [14]:
print("Data cleaning checks shown below.")
print("Model base shape:", df_model_base.shape)
print("GL accounts shape:", gl_accounts_clean.shape)
print("GL entries shape:", gl_entries_clean.shape)

display(quality_report)
display(missing_after_cleaning)
display(df_model_base.head())
display(gl_account_summary_clean.head())

Data cleaning checks shown below.
Model base shape: (73320, 74)
GL accounts shape: (21, 9)
GL entries shape: (170, 12)


,missing_count,missing_percentage,data_type
Month,73320,100.000000,object
Year,73320,100.000000,object
SalesReturnNote_Number,73313,99.990453,object
SalesCreditNote_Number,71579,97.625477,object
SalesInvoice_Number,19169,26.144299,object
SalesInvoice_Branch,19169,26.144299,object
SalesOrder_Number,18465,25.184124,object
SalesDeliveryNote_Number,18464,25.182761,object
SalesDeliveryNote_Branch,18464,25.182761,object
SalesDeliveryNote_Branch_1,18464,25.182761,object


,missing_count,missing_percentage,data_type
Date,0,0.0,object
Month,0,0.0,object
Year,0,0.0,object
SalesDeliveryNote_Branch,0,0.0,object
Customer,0,0.0,object
Customer_Name,0,0.0,object
Product,0,0.0,object
Product_ManufacturerProductCode,0,0.0,object
Contract_Number,0,0.0,object
SourceTransactionType,0,0.0,object


,Date,Month,Year,SalesDeliveryNote_Branch,Customer,Customer_Name,Product,Product_ManufacturerProductCode,Contract_Number,SourceTransactionType,Contract_D_ContractNumber,SalesInvoice_Number,SalesCreditNote_Number,SalesDeliveryNote_Number,SalesReturnNote_Number,SalesOrder_Number,Status,UnitClaimAmount,TotalClaimAmount,SalesDeliveryNoteLine_Quantity,SalesInvoiceLine_Quantity,SalesInvoice_Branch,SalesDeliveryNoteLine_DiscountPercentage,Product_Category,SalesCreditNoteLine_D_InvoiceCost,SalesCreditNoteLine_D_FixedCost,SalesDeliveryNoteLine_D_InvoiceCost,SalesDeliveryNoteLine_D_FixedCost,SalesInvoiceLine_D_InvoiceCost,SalesInvoiceLine_D_FixedCost,SalesDeliveryNote_Branch_1,SalesDeliveryNote_NetAmountLessDiscountBase,Product_CSQL_InvoiceCostForbranch,Product_CSQL_ListPriceForBranch,Product_CALC_FixedCostForBranch,Contract_Description,Contract_Expression,Contract_ValidFrom,Contract_ValidTo,Contract_Supplier_Code,Contract_Supplier_Name,claim_line_id,claim_date,customer_date,customer_year,customer_month,credit_due_value,credit_received_value,outstanding_credit_value,recovered_credit_rate,supplier_reference_value,supplier_reference_match_status,credit_recovery_status,days_outstanding,ageing_band,requires_review,review_reason,Month_was_missing,Year_was_missing,SalesDeliveryNote_Branch_was_missing,Product_ManufacturerProductCode_was_missing,Contract_Number_was_missing,Contract_D_ContractNumber_was_missing,SalesInvoice_Number_was_missing,SalesCreditNote_Number_was_missing,SalesDeliveryNote_Number_was_missing,SalesReturnNote_Number_was_missing,SalesOrder_Number_was_missing,SalesInvoice_Branch_was_missing,SalesDeliveryNote_Branch_1_was_missing,Contract_Description_was_missing,Contract_Expression_was_missing,Contract_Supplier_Code_was_missing,Contract_Supplier_Name_was_missing
0,26/02/2024,Unknown,Unknown,02,M3M500,M3 Mechanical Limited,YXS1228,38300,0086,SL/Del,039-C-0692,Unknown,Unknown,SO0000413/1,Unknown,SO0000413,Claimed,1.366861,8.2012,6,0,Unknown,69,PRSCU,0,0,5.4894,2.6953,0,0,02,392.18,6.235348,14.21,4.61415752,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,1,2024-02-26,2024-02-26,2024,2,8.2012,0.0,8.2012,0.0,8.2012,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value,1,1,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0
1,26/02/2024,Unknown,Unknown,02,M3M500,M3 Mechanical Limited,YXS12S28,38322,0086,SL/Del,039-C-0692,Unknown,Unknown,SO0000413/1,Unknown,SO0000413,Claimed,1.783139,3.5663,2,0,Unknown,69,PRSCU,0,0,7.1612,3.5162,0,0,02,392.18,8.130964,18.53,6.01691336,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,2,2024-02-26,2024-02-26,2024,2,3.5663,0.0,3.5663,0.0,3.5663,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value,1,1,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0
2,26/02/2024,Unknown,Unknown,02,M3M500,M3 Mechanical Limited,YXS62822,38204,0086,SL/Del,039-C-0692,Unknown,Unknown,SO0000413/1,Unknown,SO0000413,Claimed,1.190942,2.3819,2,0,Unknown,69,PRSCU,0,0,4.7829,2.3484,0,0,02,392.18,5.432344,12.38,4.01993456,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,3,2024-02-26,2024-02-26,2024,2,2.3819,0.0,2.3819,0.0,2.3819,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value,1,1,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0
3,26/02/2024,Unknown,Unknown,02,M3M500,M3 Mechanical Limited,YXS252815,38492,0086,SL/Del,039-C-0692,Unknown,Unknown,SO0000413/1,Unknown,SO0000413,Claimed,2.488979,9.9559,4,0,Unknown,69,PRSCU,0,0,9.9959,4.908,0,0,02,392.18,11.351756,25.87,8.40029944,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,4,2024-02-26,2024-02-26,2024,2,9.9559,0.0,9.9559,0.0,9.9559,Derived from SupportData.xlsx,Outstanding,928,365+,True,Outstanding credit value,1,1,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0
4,26/02/2024,Unknown,Unknown,02,M3M500,M3 Mechanical Limited,YXS252215,38490,0086,SL/Del,039-C-0692,Unknown,Unknown,SO0000413/1,Unknown,SO0000413,Claimed,0.890474,5.3428,6,0,Unknown,69,PRSCU,0,0,3.5762

,gl_account_code,gl_account_description,gl_supplier_key,gl_entry_rows,gl_total_credit_amount,gl_first_entry_date,gl_last_entry_date,gl_current_balance,gl_current_balance_abs,gl_has_entry_detail
0,01-85001,Supplier13 E071 Claimbacks,SUPPLIER13,105,242784.06,2026-01-05,2026-03-07,-242784.06,242784.06,1
1,01-85002,Supplier2 E005 Claimbacks,SUPPLIER2,1,6240.00,NaT,NaT,-6240.00,6240.00,1
2,01-85007,Supplier9 E022 Claimbacks,SUPPLIER9,5,5774.89,2026-04-06,2026-10-07,-5774.89,5774.89,1
3,01-85017,Supplier3 E055 Claimbacks,SUPPLIER3,4,5550.00,2026-10-04,2026-10-04,-5550.00,5550.00,1
4,01-85021,Supplier19 E070 Claimbacks,SUPPLIER19,4,9124.28,NaT,NaT,-9124.28,9124.28,1


Save datasets into cleaned cells for future use


In [15]:
clean_data_dir.mkdir(parents=True, exist_ok=True)

support_cleaned = df.copy()
gl_accounts_cleaned = gl_accounts_clean.copy()
gl_entries_cleaned = gl_entries_clean.copy()

support_cleaned.to_csv(support_cleaned_file, index=False)
gl_accounts_cleaned.to_csv(gl_accounts_cleaned_file, index=False)
gl_entries_cleaned.to_csv(gl_entries_cleaned_file, index=False)

print("Saved support_cleaned:", support_cleaned_file)
print("Saved gl_accounts_cleaned:", gl_accounts_cleaned_file)
print("Saved gl_entries_cleaned:", gl_entries_cleaned_file)
print("support_cleaned shape:", support_cleaned.shape)
print("gl_accounts_cleaned shape:", gl_accounts_cleaned.shape)
print("gl_entries_cleaned shape:", gl_entries_cleaned.shape)


Saved support_cleaned: outputs\cleaned_data\support_cleaned.csv
Saved gl_accounts_cleaned: outputs\cleaned_data\gl_accounts_cleaned.csv
Saved gl_entries_cleaned: outputs\cleaned_data\gl_entries_cleaned.csv
support_cleaned shape: (73320, 57)
gl_accounts_cleaned shape: (21, 9)
gl_entries_cleaned shape: (170, 12)
